# Explore latest datasets snapshot

Tinkering notebook to decide which fields the slim historic files should keep
(see README "Open decisions"). Reads columns **remotely** via `HfFileSystem` +
parquet column projection — only the requested columns are transferred, not the
full 377 MB file.

## SetUp

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from dotenv import load_dotenv
from huggingface_hub import HfFileSystem

load_dotenv()
fs = HfFileSystem()

LATEST = "2026-07-15"  # last week in the snapshot repo as of 2026-07-22
PATH = f"datasets/hfmlsoc/hub_weekly_snapshots/datasets/{LATEST}/datasets.parquet"

## Schema + per-column size (footer only, no data transfer)

In [2]:
from collections import defaultdict

f = pq.ParquetFile(fs.open(PATH))
md = f.metadata
sizes = defaultdict(int)
for rg in range(md.num_row_groups):
    g = md.row_group(rg)
    for ci in range(g.num_columns):
        col = g.column(ci)
        sizes[col.path_in_schema.split(".")[0]] += col.total_compressed_size

print(f"rows: {md.num_rows:,} | total compressed: {sum(sizes.values())/1e6:.0f} MB")
for name in f.schema_arrow.names:
    print(f"{name:20s} {str(f.schema_arrow.field(name).type):25s} {sizes[name]/1e6:8.1f} MB")

rows: 963,191 | total compressed: 377 MB
_id                  large_string                  16.2 MB
id                   large_string                  20.5 MB
author               large_string                   4.6 MB
cardData             large_string                 204.4 MB
disabled             bool                           0.0 MB
gated                large_string                   0.1 MB
lastModified         timestamp[us]                  8.1 MB
likes                int64                          0.2 MB
trendingScore        double                         0.0 MB
private              bool                           0.0 MB
sha                  large_string                  39.3 MB
description          large_string                  56.2 MB
downloads            int64                          1.4 MB
downloadsAllTime     int64                          1.9 MB
mainSize             double                         6.9 MB
tags                 list<element: string>          8.4 MB
createdAt      

## Load selected columns

Edit `COLS` to pull whatever you want to inspect. Avoid `cardData`/`description`/`sha`
unless needed — they are the heavy ones (204/56/39 MB).

In [3]:
COLS = ["_id", "id", "author", "likes", "downloads", "downloadsAllTime",
        "trendingScore", "tags", "mainSize", "gated", "createdAt"]

df = f.read(columns=COLS).to_pandas()
df.head()

,_id,id,author,likes,downloads,downloadsAllTime,trendingScore,tags,mainSize,gated,createdAt
0,6a292cbbe1b5c7903e6fbe30,openbmb/UltraX-Preview,openbmb,219,2307,2312,160.0,"[task_categories:text-generation, language:en,...",4.869155e+11,False,2026-06-10 09:22:03
1,6a4cc0ac90ce9cc602189d11,FlyRank/internship-warehouse,FlyRank,270,1898,1898,67.0,"[language:en, license:other, size_categories:1...",1.168719e+09,auto,2026-07-07 09:02:36
2,6a437ed52e089285573dcfd3,markov-ai/gaming-500-hours,markov-ai,179,31283,31283,45.0,"[size_categories:n<1K, format:json, modality:t...",1.598372e+12,False,2026-06-30 08:31:17
3,69f68e2f5ec43b12d4e2735f,LiquidAI/antidoom-mix-v1.0,LiquidAI,100,786,839,42.0,"[task_categories:text-generation, language:en,...",5.979998e+08,False,2026-05-02 23:52:15
4,69df2c30f5f5a426fc2ba699,AlicanKiraz0/Turkce-Atlas-Instruct,AlicanKiraz0,40,209,212,40.0,"[task_categories:text-generation, task_categor...",5.108912e+08,False,2026-04-15 06:12:00


## Open question: paper references

`paperswithcode_id` is not the only paper carrier — `tags` contains `arxiv:XXXX.XXXXX`
entries and `citation` holds free-text BibTeX. Run this to compare coverage:

In [4]:
t = f.read(columns=["tags", "paperswithcode_id", "citation"])
n = t.num_rows
arxiv = sum(1 for row in t.column("tags").to_pylist()
            if row and any(x.startswith("arxiv:") for x in row))
pwc = n - t.column("paperswithcode_id").null_count
cit = sum(1 for c in t.column("citation").to_pylist() if c)
print(f"rows {n:,}")
print(f"arxiv tag:          {arxiv:8,} ({arxiv/n:.1%})")
print(f"paperswithcode_id:  {pwc:8,} ({pwc/n:.1%})")
print(f"citation non-empty: {cit:8,} ({cit/n:.1%})")

rows 963,191
arxiv tag:            33,049 (3.4%)
paperswithcode_id:     1,730 (0.2%)
citation non-empty:    4,662 (0.5%)
